In [0]:
%sql
select * from order_lines_messy limit 10;

In [0]:
%sql
select * from products_ref limit 10;

In [0]:
%sql
select * from clients_messy limit 10;

## Data Processing Pipeline - Step 1

This notebook:
* Loads order_lines_messy, products_ref, and clients_messy tables
* Fills missing unit_price values using products_ref base_price
* Cleans client information (removes prefixes and suffixes)
* Creates a final clean table with complete order and client information

In [0]:
# Read the three tables
order_lines_df = spark.table("order_lines_messy")
products_df = spark.table("products_ref")
clients_df = spark.table("clients_messy")

print(f"Order lines: {order_lines_df.count()} rows")
print(f"Products: {products_df.count()} rows")
print(f"Clients: {clients_df.count()} rows")

In [0]:
from pyspark.sql.functions import coalesce, col

# Join order_lines with products to get base_price
order_lines_with_price = order_lines_df.join(
    products_df.select("product_id", "base_price"),
    on="product_id",
    how="left"
)

# Fill missing unit_price with base_price
order_lines_clean = order_lines_with_price.withColumn(
    "unit_price",
    coalesce(col("unit_price"), col("base_price"))
).drop("base_price")

# Check for any remaining nulls
missing_prices = order_lines_clean.filter(col("unit_price").isNull()).count()
print(f"Rows with missing unit_price after fill: {missing_prices}")

display(order_lines_clean.limit(10))

In [0]:
from pyspark.sql.functions import trim, regexp_replace

# Clean first_name: remove **, tabs, and extra spaces
clients_clean = clients_df.withColumn(
    "first_name",
    trim(regexp_replace(col("first_name"), r"^[\*\*\t\s]+", ""))
)

# Clean last_name: trim spaces
clients_clean = clients_clean.withColumn(
    "last_name",
    trim(col("last_name"))
)

# Clean email: remove !!! and other special suffixes
clients_clean = clients_clean.withColumn(
    "email",
    regexp_replace(col("email"), r"[!]+$", "")
)

display(clients_clean.limit(10))

In [0]:
# Join order lines with products (to get product names)
final_df = order_lines_clean.join(
    products_df.select("product_id", "product_name"),
    on="product_id",
    how="left"
)

# Join with clean client information
final_df = final_df.join(
    clients_clean,
    on="client_id",
    how="left"
)

# Select and order columns
final_df = final_df.select(
    "line_id",
    "order_id",
    "client_id",
    "first_name",
    "last_name",
    "email",
    "product_id",
    "product_name",
    "quantity",
    "unit_price"
)

print(f"Final dataset: {final_df.count()} rows")
display(final_df.limit(20))

In [0]:
# Save to a new table
final_table_name = "order_lines_clean"

final_df.write.format("delta").mode("overwrite").saveAsTable(final_table_name)

print(f"✅ Cleaned data saved to table: {final_table_name}")
print(f"Total rows: {spark.table(final_table_name).count()}")

In [0]:
%sql
